# FraudShield Advanced Lab: Built-in Algorithms and HPO

## Scenario

You are a machine learning engineer on the FraudShield team. Your task is to:

1. **Train an XGBoost model** in Algorithm Mode on FraudShield transaction data.
2. **Train a K-Means clustering model** on the same feature set (unsupervised) to
   identify transaction segments.
3. **Optimize XGBoost with Hyperparameter Optimization (HPO)** using Bayesian strategy
   to find the best-performing configuration.
4. **Compare results** across models and HPO trials.

All training jobs use `ml.m5.xlarge` instances (Free Tier eligible). This notebook
runs on SageMaker Studio `ml.t3.medium`.

Fill in the `TODO` sections to complete each step. Present your results at each
checkpoint.

In [ ]:
%pip install -q sagemaker boto3 pandas numpy

In [ ]:
import time
import json
import os
import pandas as pd
import numpy as np
import boto3
import sagemaker
from sagemaker import image_uris
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.tuner import (
    HyperparameterTuner,
    ContinuousParameter,
    IntegerParameter,
)

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session.boto_region_name
bucket = sagemaker_session.default_bucket()
prefix = "fraudshield-lab-algorithms"

print(f"Role:    {role}")
print(f"Region:  {region}")
print(f"Bucket:  {bucket}")

## Concept Connection: Built-in Algorithms

SageMaker provides **optimized, pre-built algorithm containers** that you can use without
writing any training code. The key advantages:

- Containers are maintained, tested, and optimized by AWS for distributed training.
- You retrieve the container image URI with `sagemaker.image_uris.retrieve()` and pass it
  directly to an `Estimator`.
- All algorithm behavior is controlled through hyperparameters -- no `entry_point` script.
- Data is provided in specific formats (CSV with target in column 0 for XGBoost,
  headerless CSV or RecordIO-protobuf for K-Means).

This approach is called **Algorithm Mode** and is distinct from **Script Mode**, where you
supply custom Python training code.

In [ ]:
# Generate synthetic FraudShield data for the lab
np.random.seed(42)
n_samples = 2000

amounts = np.random.exponential(200, n_samples)
hours = np.random.randint(0, 24, n_samples)
distances = np.random.exponential(100, n_samples)
merchant_risk = np.random.beta(2, 5, n_samples)
is_intl = np.random.binomial(1, 0.15, n_samples)
velocity = np.random.poisson(3, n_samples)
card_age = np.random.randint(1, 3650, n_samples)

fraud_score = (
    0.3 * (amounts > 500).astype(float)
    + 0.2 * ((hours < 5) | (hours > 22)).astype(float)
    + 0.2 * (distances > 300).astype(float)
    + 0.15 * (merchant_risk > 0.5).astype(float)
    + 0.1 * is_intl
    + 0.05 * (velocity > 6).astype(float)
)
is_fraud = (fraud_score > np.random.uniform(0, 1, n_samples) * 1.5).astype(int)

# Target column first, no header (Algorithm Mode CSV format)
full_df = pd.DataFrame({
    "is_fraud": is_fraud,
    "amount": np.round(amounts, 2),
    "hour": hours,
    "distance_from_home": np.round(distances, 1),
    "merchant_risk_score": np.round(merchant_risk, 4),
    "is_international": is_intl,
    "transaction_velocity_1h": velocity,
    "card_age_days": card_age,
})

split = int(0.8 * n_samples)
train_df = full_df.iloc[:split]
val_df = full_df.iloc[split:]

train_df.to_csv("train.csv", index=False, header=False)
val_df.to_csv("validation.csv", index=False, header=False)

train_s3 = sagemaker_session.upload_data("train.csv", bucket=bucket, key_prefix=f"{prefix}/data/train")
val_s3 = sagemaker_session.upload_data("validation.csv", bucket=bucket, key_prefix=f"{prefix}/data/validation")

print(f"Train uploaded to: {train_s3}")
print(f"Validation uploaded to: {val_s3}")
print(f"Train shape: {train_df.shape}, Val shape: {val_df.shape}")
print(f"Fraud rate: {is_fraud.mean():.2%}")

---
## Step 1: Train XGBoost in Algorithm Mode

Use `sagemaker.image_uris.retrieve()` to get the XGBoost container, create an `Estimator`
with appropriate hyperparameters, and launch a training job.

In [ ]:
# TODO: Retrieve the XGBoost container image URI
xgb_image = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.5-1",
)

# TODO: Create the Estimator (Algorithm Mode -- no entry_point)
xgb_estimator = Estimator(
    image_uri=xgb_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/xgboost/output",
    sagemaker_session=sagemaker_session,
)

# TODO: Set hyperparameters -- choose values for the following:
#   objective, eval_metric, num_round, max_depth, eta
xgb_estimator.set_hyperparameters(
    objective="binary:logistic",        # TODO: verify this is the right objective
    eval_metric="auc",                  # TODO: verify this is the right metric
    num_round=100,                      # TODO: choose a reasonable number of rounds
    max_depth=5,                        # TODO: choose max tree depth
    eta=0.2,                            # TODO: choose learning rate
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=10,
)

print("XGBoost Estimator ready.")

In [ ]:
train_input = TrainingInput(train_s3, content_type="text/csv")
val_input = TrainingInput(val_s3, content_type="text/csv")

xgb_estimator.fit(
    inputs={"train": train_input, "validation": val_input},
    job_name=f"fraudshield-lab-xgb-{int(time.time())}",
    wait=True,
    logs="All",
)

print(f"\nXGBoost training complete.")
print(f"Job name: {xgb_estimator.latest_training_job.name}")
print(f"Model artifact: {xgb_estimator.model_data}")

---
## Step 2: Train K-Means for Transaction Segmentation

K-Means is an unsupervised algorithm. We drop the target column and cluster transactions
by their feature similarity. This can reveal behavioral segments useful as downstream
features.

In [ ]:
# Prepare feature-only data (drop target column)
feature_cols = ["amount", "hour", "distance_from_home", "merchant_risk_score",
                "is_international", "transaction_velocity_1h", "card_age_days"]

kmeans_df = full_df[feature_cols].copy()
kmeans_df.to_csv("kmeans_data.csv", index=False, header=False)

kmeans_s3 = sagemaker_session.upload_data(
    "kmeans_data.csv", bucket=bucket, key_prefix=f"{prefix}/kmeans/data"
)

# TODO: Retrieve the K-Means container image
kmeans_image = image_uris.retrieve(
    framework="kmeans",
    region=region,
)

# TODO: Create the K-Means Estimator and set hyperparameters
kmeans_estimator = Estimator(
    image_uri=kmeans_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/kmeans/output",
    sagemaker_session=sagemaker_session,
)

kmeans_estimator.set_hyperparameters(
    k=5,                   # TODO: choose number of clusters
    feature_dim=7,
    mini_batch_size=200,
    max_iterations=100,
    epochs=10,
)

print(f"K-Means Estimator ready. k=5, feature_dim=7")

In [ ]:
kmeans_input = TrainingInput(kmeans_s3, content_type="text/csv")

kmeans_estimator.fit(
    inputs={"train": kmeans_input},
    job_name=f"fraudshield-lab-kmeans-{int(time.time())}",
    wait=True,
    logs="All",
)

print(f"\nK-Means training complete.")
print(f"Job name: {kmeans_estimator.latest_training_job.name}")
print(f"Model artifact: {kmeans_estimator.model_data}")

---
## Presentation Checkpoint: Compare XGBoost and K-Means

Navigate to the **SageMaker Console** and compare the two training jobs:

1. **Training > Training jobs** -- Find both jobs by their names printed above.
2. Compare:
   - **Instance type and training duration** for each job.
   - **XGBoost metrics** (validation:auc) -- how well does the model discriminate fraud?
   - **K-Means** -- note the loss metric (within-cluster sum of squared distances).
3. Consider: How might K-Means cluster assignments be used as an additional feature
   for the XGBoost fraud detection model?

---
## Concept Connection: HPO Strategies

Hyperparameter Optimization (HPO) automates the search for the best hyperparameter
configuration. SageMaker supports two primary strategies:

- **Bayesian Optimization**: Maintains a probabilistic model (surrogate) of the objective
  function. After each trial completes, the surrogate is updated and used to select the
  next most promising configuration. This approach **learns from previous trials** and
  converges faster, but benefits from sequential execution.

- **Random Search**: Samples hyperparameters uniformly from the specified ranges. Trials
  are independent, so they can all run in parallel. This is less sample-efficient but
  covers the search space **uniformly** and is useful when you have budget for many
  parallel jobs.

The key trade-off: **Bayesian + low parallelism = higher quality per trial**;
**Random + high parallelism = faster wall-clock time**.

---
## Step 3: Optimize XGBoost with HPO

Define hyperparameter search ranges, create a `HyperparameterTuner`, and launch
a tuning job that will run up to 10 trials with Bayesian strategy.

In [ ]:
# TODO: Define hyperparameter ranges for the tuner
hyperparameter_ranges = {
    "eta": ContinuousParameter(0.01, 0.3),
    "max_depth": IntegerParameter(3, 10),
    "min_child_weight": IntegerParameter(1, 10),
    "subsample": ContinuousParameter(0.5, 1.0),
    "colsample_bytree": ContinuousParameter(0.5, 1.0),
    "num_round": IntegerParameter(50, 300),
}

# TODO: Create a base estimator for the tuner
hpo_base = Estimator(
    image_uri=xgb_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/hpo/output",
    sagemaker_session=sagemaker_session,
)

hpo_base.set_hyperparameters(
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=10,
)

# TODO: Create the HyperparameterTuner
tuner = HyperparameterTuner(
    estimator=hpo_base,
    objective_metric_name="validation:auc",
    hyperparameter_ranges=hyperparameter_ranges,
    objective_type="Maximize",
    max_jobs=10,
    max_parallel_jobs=2,
    strategy="Bayesian",
)

print("HyperparameterTuner configured:")
print(f"  Strategy:      Bayesian")
print(f"  Max jobs:      10")
print(f"  Max parallel:  2")
print(f"  Objective:     Maximize validation:auc")

In [ ]:
tuner.fit(
    inputs={"train": train_input, "validation": val_input},
    job_name=f"fraudshield-lab-hpo-{int(time.time())}",
    wait=True,
)

print(f"HPO job complete: {tuner.latest_tuning_job.name}")

In [ ]:
sm = boto3.client("sagemaker")
tuning_job_name = tuner.latest_tuning_job.name

resp = sm.describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuning_job_name
)

best = resp["BestTrainingJob"]
best_job_name = best["TrainingJobName"]
best_metric = best["FinalHyperParameterTuningJobObjectiveMetric"]["Value"]

print(f"Best training job: {best_job_name}")
print(f"Best validation AUC: {best_metric}")

best_desc = sm.describe_training_job(TrainingJobName=best_job_name)
best_hps = best_desc["HyperParameters"]

print("\nBest hyperparameters:")
for hp in sorted(hyperparameter_ranges.keys()):
    print(f"  {hp:25s}  {best_hps.get(hp, 'N/A')}")

# Compare with baseline XGBoost
print(f"\nBaseline XGBoost job: {xgb_estimator.latest_training_job.name}")
baseline_desc = sm.describe_training_job(
    TrainingJobName=xgb_estimator.latest_training_job.name
)
for metric in baseline_desc.get("FinalMetricDataList", []):
    if "auc" in metric["MetricName"].lower():
        print(f"Baseline validation AUC: {metric['Value']}")

print(f"HPO best validation AUC:  {best_metric}")

In [ ]:
analytics = tuner.analytics()
trials_df = analytics.dataframe()

print(f"Total HPO trials: {len(trials_df)}")

trials_sorted = trials_df.sort_values("FinalObjectiveValue", ascending=False)
display(trials_sorted.head(10))

---
## Presentation Checkpoint: HPO Results

Review the HPO results and prepare to discuss:

1. **Best trial vs baseline**: How much did HPO improve validation AUC over your
   manually chosen hyperparameters?
2. **Hyperparameter sensitivity**: Which hyperparameters varied the most across the
   top-performing trials? Which appeared to matter least?
3. **Bayesian convergence**: Did later trials tend to perform better than earlier ones?
   This would indicate the Bayesian surrogate model was learning effectively.
4. **Cost consideration**: Each trial ran on `ml.m5.xlarge`. With 10 trials and
   max 2 parallel, estimate the total training time and cost.

---
## Cleanup

Remove all S3 artifacts and local temporary files created during this lab.

In [ ]:
s3 = boto3.resource("s3")
s3_bucket = s3.Bucket(bucket)

for sub in ["data", "xgboost", "kmeans", "hpo"]:
    full_prefix = f"{prefix}/{sub}/"
    objects = list(s3_bucket.objects.filter(Prefix=full_prefix))
    if objects:
        s3_bucket.delete_objects(
            Delete={"Objects": [{"Key": obj.key} for obj in objects]}
        )
        print(f"Deleted {len(objects)} objects under s3://{bucket}/{full_prefix}")

for f in ["train.csv", "validation.csv", "kmeans_data.csv"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed local file: {f}")

print("Cleanup complete.")